# Step 1 — Preprocessing

Reads the raw dataset from `../../data/raw_data/diabetes_dataset.csv`, applies quality filters, and outputs a clean 7-column `final_df` Parquet to `../../data/final_df`.

> Neither the raw CSV nor the output Parquet are committed to the repository. The CSV must be downloaded to `data/raw_data/` before running this notebook.

**Feature set (clean, no redundancy):**

| Column | Type | Filter |
|---|---|---|
| `hbA1c_level` | double | 3.5 – 15.0 |
| `blood_glucose_level` | integer | 20 – 600 |
| `bmi` | double | 10.0 – 80.0 |
| `age` | double | 0 – 120 |
| `hypertension` | integer | 0 or 1 |
| `heart_disease` | integer | 0 or 1 |
| `diabetes` | integer | 0 or 1 (label) |

**Pipeline position:** this output feeds directly into `02_feature_engineering.py`. On GCP, upload the output to `gs://team10-diabetes-data/processed/final_df` before submitting the distributed jobs.

### 1. Spark Session

In [ ]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import monotonically_increasing_id

spark = SparkSession.builder \
    .appName("DiabetesPreprocessing") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", True) \
    .getOrCreate()

### 2. Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))

INPUT_CSV       = os.path.join(NOTEBOOK_DIR, "..", "..", "data", "raw_data", "diabetes_dataset.csv")
OUTPUT_FINAL_DF = os.path.join(NOTEBOOK_DIR, "..", "..", "data", "df_processed")

print("Input :", os.path.normpath(INPUT_CSV))
print("Output:", os.path.normpath(OUTPUT_FINAL_DF))

### 3. Data Ingestion

In [30]:
df_raw = spark.read.csv(INPUT_CSV, header=True, inferSchema=True)

df_raw.printSchema()
df_raw.show(5)
print("Row count:", df_raw.count())

root
 |-- year: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- location: string (nullable = true)
 |-- race:AfricanAmerican: integer (nullable = true)
 |-- race:Asian: integer (nullable = true)
 |-- race:Caucasian: integer (nullable = true)
 |-- race:Hispanic: integer (nullable = true)
 |-- race:Other: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- heart_disease: integer (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- hbA1c_level: double (nullable = true)
 |-- blood_glucose_level: integer (nullable = true)
 |-- diabetes: integer (nullable = true)

+----+------+----+--------+--------------------+----------+--------------+-------------+----------+------------+-------------+---------------+-----+-----------+-------------------+--------+
|year|gender| age|location|race:AfricanAmerican|race:Asian|race:Caucasian|race:Hispanic|race:Other|hypertension|h

### 4. Data Cleaning & Quality Filter

In [31]:
REQUIRED_COLS = ["hbA1c_level", "blood_glucose_level", "bmi", "age",
                 "hypertension", "heart_disease", "diabetes"]

df_clean = df_raw.na.drop(subset=REQUIRED_COLS)

df_filtered = df_clean.filter(
    (F.col("diabetes").isin(0, 1))                    &
    (F.col("hypertension").isin(0, 1))                &
    (F.col("heart_disease").isin(0, 1))               &
    (F.col("bmi").between(10.0, 80.0))                &
    (F.col("age").between(0.0, 120.0))                &
    (F.col("hbA1c_level").between(3.5, 15.0))         &
    (F.col("blood_glucose_level").between(20, 600))
)

before = df_raw.count()
after  = df_filtered.count()
print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Dropped:     {before - after}")

Rows before: 100000
Rows after:  99991
Dropped:     9


### 5. Preview Clean Feature Set

In [32]:
df_filtered.select(*REQUIRED_COLS).show(10, truncate=True)
df_filtered.select(*REQUIRED_COLS).describe().show()

+-----------+-------------------+-----+----+------------+-------------+--------+
|hbA1c_level|blood_glucose_level|  bmi| age|hypertension|heart_disease|diabetes|
+-----------+-------------------+-----+----+------------+-------------+--------+
|        5.0|                100|27.32|32.0|           0|            0|       0|
|        5.0|                 90|19.95|29.0|           0|            0|       0|
|        4.8|                160|23.76|18.0|           0|            0|       0|
|        4.0|                159|27.32|41.0|           0|            0|       0|
|        6.5|                 90|23.75|52.0|           0|            0|       0|
|        5.7|                159|27.32|66.0|           0|            0|       0|
|        5.7|                 80|24.34|49.0|           0|            0|       0|
|        5.0|                155|20.98|15.0|           0|            0|       0|
|        6.0|                100|38.14|51.0|           0|            0|       0|
|        5.7|               

### 6. Validation — Class & Feature Distributions

In [33]:
print("Class balance:")
df_filtered.groupBy("diabetes") \
    .agg(F.count("*").alias("count")) \
    .orderBy("diabetes") \
    .show()

diabetic = df_filtered.filter(F.col("diabetes") == 1)

print("Hypertension (diabetic patients):")
diabetic.groupBy("hypertension") \
    .agg(F.count("*").alias("count")) \
    .orderBy("hypertension").show()

print("Heart Disease (diabetic patients):")
diabetic.groupBy("heart_disease") \
    .agg(F.count("*").alias("count")) \
    .orderBy("heart_disease").show()

print("HbA1c & Blood Glucose stats (diabetic vs non-diabetic):")
df_filtered.groupBy("diabetes") \
    .agg(
        F.round(F.mean("hbA1c_level"), 2).alias("avg_hbA1c"),
        F.round(F.mean("blood_glucose_level"), 2).alias("avg_blood_glucose"),
        F.round(F.mean("bmi"), 2).alias("avg_bmi"),
        F.round(F.mean("age"), 2).alias("avg_age"),
    ) \
    .orderBy("diabetes") \
    .show()

Class balance:
+--------+-----+
|diabetes|count|
+--------+-----+
|       0|91494|
|       1| 8497|
+--------+-----+

Hypertension (diabetic patients):
+------------+-----+
|hypertension|count|
+------------+-----+
|           0| 6409|
|           1| 2088|
+------------+-----+

Heart Disease (diabetic patients):
+-------------+-----+
|heart_disease|count|
+-------------+-----+
|            0| 7230|
|            1| 1267|
+-------------+-----+

HbA1c & Blood Glucose stats (diabetic vs non-diabetic):
+--------+---------+-----------------+-------+-------+
|diabetes|avg_hbA1c|avg_blood_glucose|avg_bmi|avg_age|
+--------+---------+-----------------+-------+-------+
|       0|      5.4|           132.85|  26.88|  40.12|
|       1|     6.94|           194.09|  31.97|  60.95|
+--------+---------+-----------------+-------+-------+



### 7. Save `final_df`

Narrow to the 7 columns consumed by `02_feature_engineering.py`.

In [ ]:
final_df = df_filtered \
    .withColumn("row_id", monotonically_increasing_id()) \
    .select("row_id", *REQUIRED_COLS)

final_df.show(10, truncate=False)

final_df.write.mode("overwrite").parquet(OUTPUT_FINAL_DF)
print(f"Saved {final_df.count()} rows → {os.path.normpath(OUTPUT_FINAL_DF)}")

### 8. Stop Spark

In [35]:
spark.stop()
print("Done. Next step: feature_engineering.py")

Done. Next step: feature_engineering.py
